[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DavinciDreams/SymbioGPT/blob/main/fuse_juliaslm.ipynb)

# JuliaSLM → JuliaFluxGPT: Symbiogenesis Projection Fusion

Transfer learned representations from **JuliaSLM** (d=256, 6L, 4H MHA, val_loss=3.54, curated data)
into the **JuliaFluxGPT** architecture (d=512, 8L, 8Q/2KV GQA, ~23M params) using
symbiogenesis-style projection fusion, then fine-tune on the curated philosophy corpus.

**Key insight**: JuliaSLM learned good representations on curated data (val_loss=3.54 at ~5M params)
but is limited by model capacity. JuliaFluxGPT has capacity (~23M) but was trained on dirty data
(val_loss ~6.6). By projecting JuliaSLM's weights into the larger architecture and fine-tuning
on curated data, we get the best of both worlds.

| | JuliaSLM (source) | JuliaFluxGPT (target) |
|---|---|---|
| d_model | 256 | 512 |
| Layers | 6 | 8 |
| Attention | 4H MHA | 8Q/2KV GQA |
| FFN | SwiGLU 640 | SwiGLU 1344 |
| Vocab | 2000 BPE | 2000 BPE |
| Val Loss | 3.54 (curated) | ~6.6 (dirty) |
| Params | ~5M | ~23M |

**Projection strategy** (from symbiogenesis `fusion.py`):
1. Zero-pad all weight matrices from d=256 → d=512
2. Head-aware Q duplication: 4 MHA heads → 8 GQA query heads (each src head duplicated 2×)
3. KV head merging: avg pairs of src K/V heads → 2 target KV heads
4. FFN zero-pad: inner_dim 640 → 1344
5. Copy first 6 layers, leave layers 7-8 randomly initialized
6. Fine-tune on 266M curated tokens

GitHub: https://github.com/DavinciDreams/SymbioGPT

In [ ]:
# 1. Setup
!pip install -q wandb huggingface_hub
!git clone https://github.com/DavinciDreams/SymbioGPT.git /content/SymbioGPT 2>/dev/null || (cd /content/SymbioGPT && git pull)
%cd /content/SymbioGPT

In [ ]:
# 2. GPU check + imports
import os, sys, math, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, HfApi, create_repo

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"Memory: {mem / 1e9:.1f} GB")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 3. W&B + HF login (mandatory per project rules)
import wandb
from huggingface_hub import login as hf_login

wandb.login()
hf_login()

In [ ]:
# 4. Download data + model weights
DATA_REPO = "LisaMegaWatts/SymbioGPT-10M"
SLM_REPO = "LisaMegaWatts/JuliaSLM"
FLUX_REPO = "LisaMegaWatts/JuliaFluxGPT"
os.makedirs("data", exist_ok=True)

# Curated training data (266M train, 72M val tokens)
print("Downloading curated data...")
hf_hub_download(repo_id=DATA_REPO, filename="data/train_curated.txt.tokens.pt", local_dir=".")
hf_hub_download(repo_id=DATA_REPO, filename="data/val.txt.tokens.pt", local_dir=".")

# JuliaSLM source weights (NPZ from convert_juliaslm.jl)
print("Downloading JuliaSLM weights...")
try:
    slm_path = hf_hub_download(repo_id=SLM_REPO, filename="juliaslm_weights.npz", local_dir=".")
    print(f"  Downloaded: {slm_path}")
except Exception as e:
    print(f"  ERROR: {e}")
    print("  JuliaSLM NPZ not found on HF. You need to run the conversion first:")
    print("    1. cd JuliaSLM/")
    print("    2. huggingface-cli download LisaMegaWatts/JuliaSLM final.jld2 --local-dir .")
    print("    3. julia convert_juliaslm.jl")
    print("    4. huggingface-cli upload LisaMegaWatts/JuliaSLM juliaslm_weights.npz")
    raise

# JuliaFluxGPT target weights (for baseline comparison)
print("Downloading JuliaFluxGPT weights...")
flux_path = hf_hub_download(repo_id=FLUX_REPO, filename="juliaflux_weights.pt", local_dir=".")

# Load and chunk tokens
CTX = 256
print("\nLoading tokens...")
train_tokens = torch.load("data/train_curated.txt.tokens.pt", weights_only=True).tolist()
val_tokens = torch.load("data/val.txt.tokens.pt", weights_only=True).tolist()

def chunk(tokens, seq_len):
    n = len(tokens) // (seq_len + 1)
    tokens = tokens[:n * (seq_len + 1)]
    data = torch.tensor(tokens, dtype=torch.long).reshape(n, seq_len + 1)
    return data[:, :-1], data[:, 1:]

train_inputs, train_labels = chunk(train_tokens, CTX)
val_inputs, val_labels = chunk(val_tokens, CTX)
print(f"Train: {len(train_inputs):,} seqs ({len(train_inputs)*CTX:,} tokens)")
print(f"Val: {len(val_inputs):,} seqs ({len(val_inputs)*CTX:,} tokens)")
del train_tokens, val_tokens

In [ ]:
# 5. Model definition
sys.path.insert(0, "/content/SymbioGPT")
from juliaflux_model import JuliaFluxConfig, JuliaFluxGPT

# Target architecture (JuliaFluxGPT ~23M)
config = JuliaFluxConfig(
    d_model=512, n_layers=8, n_heads=8, n_kv_heads=2,
    head_dim=64, context_length=CTX, vocab_size=2000,
    weight_tying=True, rope_base=10000.0,
)
print(f"Target: d={config.d_model}, L={config.n_layers}, H={config.n_heads}Q/{config.n_kv_heads}KV")

# Compute FFN inner dims for both architectures
raw_inner = int(4 * config.d_model * 2 / 3)
target_inner = max(64, 64 * ((raw_inner + 32) // 64))
print(f"Target FFN inner: {target_inner}")

In [ ]:
# 6. Load JuliaSLM source weights
print("Loading JuliaSLM weights from NPZ...")
slm_data = np.load(slm_path)

# Extract hyperparams
slm_d = int(slm_data["_hp_embed_dim"][0])
slm_vocab = int(slm_data["_hp_vocab_size"][0])
slm_layers = int(slm_data["_hp_n_layers"][0])
slm_heads = int(slm_data["_hp_n_heads"][0])
slm_inner = int(slm_data["_hp_inner_dim"][0])
slm_hd = int(slm_data["_hp_head_dim"][0])

# Load weight tensors
slm_weights = {}
print(f"\nJuliaSLM: d={slm_d}, vocab={slm_vocab}, layers={slm_layers}, heads={slm_heads}, ffn={slm_inner}")
print("\nWeight structure:")
total_src = 0
for key in sorted(slm_data.files):
    if key.startswith("_hp_"):
        continue
    arr = slm_data[key]
    slm_weights[key] = torch.from_numpy(arr.copy())
    total_src += arr.size
    print(f"  {key}: {arr.shape}")
print(f"\nTotal source params: {total_src:,} ({total_src/1e6:.2f}M)")

In [ ]:
# 7. Projection fusion: JuliaSLM (d=256, 6L, 4H MHA) → JuliaFluxGPT (d=512, 8L, 8Q/2KV GQA)
#
# Based on symbiogenesis/fusion.py _project_weights() with head-aware attention mapping.
#
# Dimension expansion: zero-pad from 256 → 512.
# Head mapping:
#   Q: each src head (4) → duplicated to 2 target heads (8 total)
#   KV: avg pairs of src heads → 2 target KV heads
#   Output proj: split equally across duplicated head pairs

def project_juliaslm_to_juliafluxgpt(slm_weights, config, slm_d, slm_layers, slm_heads, slm_inner):
    """Project JuliaSLM weights into a fresh JuliaFluxGPT model.
    
    Returns the model with transferred weights.
    """
    model = JuliaFluxGPT(config)
    sd = model.state_dict()
    hd = 64  # head_dim (same for both architectures)
    kv_dim = config.n_kv_heads * hd  # 128
    transferred = 0
    
    # --- Embedding ---
    # (vocab, slm_d) → (vocab, 512), noise-pad extra dims to avoid RMSNorm distortion
    emb = slm_weights['tok_emb.weight']  # (vocab, 256)
    emb_std = emb.std().item()
    sd['tok_emb.weight'][:, :slm_d] = emb
    sd['tok_emb.weight'][:, slm_d:] = torch.randn_like(sd['tok_emb.weight'][:, slm_d:]) * emb_std * 0.02
    transferred += emb.numel()
    print(f"  Embedding: ({emb.shape[0]}, {slm_d}) → ({emb.shape[0]}, {config.d_model})")
    
    # --- Per-layer projection (6 source layers → first 6 of 8 target layers) ---
    for i in range(slm_layers):
        pfx = f'blocks.{i}'
        
        # RMSNorm: (slm_d,) → (512,), pad with 1.0 to preserve normalization
        for ln in ['ln1', 'ln2']:
            key = f'{pfx}.{ln}.weight'
            src = slm_weights[f'blocks.{i}.{ln}.weight']
            sd[key][:slm_d] = src
            sd[key][slm_d:] = 1.0  # identity scaling for non-transferred dims
            transferred += src.numel()
        
        # === ATTENTION: MHA → GQA ===
        
        # wq: (4*64, 256) → (8*64, 512) with head duplication
        # Each source head duplicated to 2 target Q heads
        src_wq = slm_weights[f'blocks.{i}.attn.wq']  # (256, 256)
        wq_key = f'{pfx}.attn.wq.weight'
        sd[wq_key].zero_()
        for j in range(slm_heads):  # 4 source heads
            head_rows = src_wq[j*hd:(j+1)*hd, :]  # (64, 256)
            sd[wq_key][2*j*hd:(2*j+1)*hd, :slm_d] = head_rows       # target head 2j
            sd[wq_key][(2*j+1)*hd:(2*j+2)*hd, :slm_d] = head_rows   # target head 2j+1
        transferred += src_wq.numel()
        
        # wk + wv → wkv: merge 4 src heads → 2 target KV heads
        # wkv layout: [K_h0(64) | K_h1(64) | V_h0(64) | V_h1(64)] = (256, 512)
        src_wk = slm_weights[f'blocks.{i}.attn.wk']  # (256, 256) = 4 heads × 64
        src_wv = slm_weights[f'blocks.{i}.attn.wv']  # (256, 256) = 4 heads × 64
        wkv_key = f'{pfx}.attn.wkv.weight'
        sd[wkv_key].zero_()
        for h in range(config.n_kv_heads):  # 2 target KV heads
            # Average pairs of source heads → 1 target KV head
            k_avg = (src_wk[2*h*hd:(2*h+1)*hd, :] + src_wk[(2*h+1)*hd:(2*h+2)*hd, :]) / 2
            v_avg = (src_wv[2*h*hd:(2*h+1)*hd, :] + src_wv[(2*h+1)*hd:(2*h+2)*hd, :]) / 2
            # K section (rows 0..kv_dim-1)
            sd[wkv_key][h*hd:(h+1)*hd, :slm_d] = k_avg
            # V section (rows kv_dim..2*kv_dim-1)
            sd[wkv_key][(kv_dim + h*hd):(kv_dim + (h+1)*hd), :slm_d] = v_avg
        transferred += src_wk.numel() + src_wv.numel()
        
        # wo → proj: (256, 256) → (512, 512) with head-aware split
        # Since Q heads are duplicated, each pair produces identical output.
        # Split output projection equally: proj_col_2j = proj_col_2j+1 = src_wo_col_j / 2
        # so that (col_2j + col_2j+1) * identical_output = col_j * output
        src_wo = slm_weights[f'blocks.{i}.attn.wo']  # (256, 256) = (d, n_heads*hd)
        proj_key = f'{pfx}.attn.proj.weight'
        sd[proj_key].zero_()
        for j in range(slm_heads):  # 4 source heads
            col_slice = src_wo[:, j*hd:(j+1)*hd]  # (256, 64)
            sd[proj_key][:slm_d, 2*j*hd:(2*j+1)*hd] = col_slice / 2
            sd[proj_key][:slm_d, (2*j+1)*hd:(2*j+2)*hd] = col_slice / 2
        transferred += src_wo.numel()
        
        # === FFN: zero-pad ===
        
        # w1 → w_gate: (slm_inner, slm_d) → (target_inner, 512)
        src_w1 = slm_weights[f'blocks.{i}.ffn.w1']  # (640, 256)
        sd[f'{pfx}.ffn.w_gate.weight'].zero_()
        sd[f'{pfx}.ffn.w_gate.weight'][:slm_inner, :slm_d] = src_w1
        transferred += src_w1.numel()
        
        # v → w_up: (slm_inner, slm_d) → (target_inner, 512)
        src_v = slm_weights[f'blocks.{i}.ffn.v']  # (640, 256)
        sd[f'{pfx}.ffn.w_up.weight'].zero_()
        sd[f'{pfx}.ffn.w_up.weight'][:slm_inner, :slm_d] = src_v
        transferred += src_v.numel()
        
        # w2 → w_down: (slm_d, slm_inner) → (512, target_inner)
        src_w2 = slm_weights[f'blocks.{i}.ffn.w2']  # (256, 640)
        sd[f'{pfx}.ffn.w_down.weight'].zero_()
        sd[f'{pfx}.ffn.w_down.weight'][:slm_d, :slm_inner] = src_w2
        transferred += src_w2.numel()
        
        print(f"  Layer {i}: wq=dup×2, wkv=avg→2kv, proj=split/2, ffn=pad ({slm_inner}→{target_inner})")
    
    # Layers 6-7: random init (no source data)
    for i in range(slm_layers, config.n_layers):
        print(f"  Layer {i}: random init (no source)")
    
    # Final RMSNorm: (slm_d,) → (512,)
    src_ln = slm_weights['ln_f.weight']
    sd['ln_f.weight'][:slm_d] = src_ln
    sd['ln_f.weight'][slm_d:] = 1.0
    transferred += src_ln.numel()
    
    model.load_state_dict(sd)
    total = sum(p.numel() for p in model.parameters())
    print(f"\nTransferred {transferred:,} source params into {total:,} target params")
    print(f"Coverage: {transferred / total * 100:.1f}% of target filled from source")
    return model

print("Projecting JuliaSLM → JuliaFluxGPT...")
print(f"  Source: d={slm_d}, {slm_layers}L, {slm_heads}H MHA, ffn={slm_inner}")
print(f"  Target: d={config.d_model}, {config.n_layers}L, {config.n_heads}Q/{config.n_kv_heads}KV, ffn={target_inner}")
print()

model = project_juliaslm_to_juliafluxgpt(
    slm_weights, config, slm_d, slm_layers, slm_heads, slm_inner
)
model = model.to(device)
print(f"\nModel on {device}, {model.num_parameters:,} params")

In [ ]:
# 8. Evaluation function + baselines

def evaluate(model, val_inputs, val_labels, batch_size=128):
    """Compute val loss and perplexity."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    dev = next(model.parameters()).device
    with torch.no_grad():
        for i in range(0, len(val_inputs), batch_size):
            batch_in = val_inputs[i:i+batch_size].to(dev)
            batch_tgt = val_labels[i:i+batch_size].to(dev)
            logits = model(batch_in)
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.float().reshape(B*T, V), batch_tgt.reshape(B*T), reduction="sum"
            )
            total_loss += loss.item()
            total_tokens += B * T
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl

# Evaluate fused model before fine-tuning
fused_loss, fused_ppl = evaluate(model, val_inputs, val_labels)
print(f"Fused model (before fine-tuning): val_loss={fused_loss:.4f} ppl={fused_ppl:.1f}")

# Load and evaluate original JuliaFluxGPT for baseline comparison
print("\nLoading original JuliaFluxGPT for comparison...")
orig_sd = torch.load(flux_path, map_location=device, weights_only=True)
orig_model = JuliaFluxGPT(config).to(device)
orig_model.load_state_dict(orig_sd, strict=False)
orig_loss, orig_ppl = evaluate(orig_model, val_inputs, val_labels)
print(f"Original JuliaFluxGPT (dirty): val_loss={orig_loss:.4f} ppl={orig_ppl:.1f}")
del orig_model, orig_sd
torch.cuda.empty_cache()

print(f"\n{'Model':<30} {'Val Loss':>10} {'PPL':>10}")
print("-" * 52)
print(f"{'JuliaSLM (source, d=256)':<30} {'3.5400':>10} {'34.5':>10}")
print(f"{'JuliaFluxGPT (dirty, d=512)':<30} {orig_loss:>10.4f} {orig_ppl:>10.1f}")
print(f"{'Fused (before fine-tune)':<30} {fused_loss:>10.4f} {fused_ppl:>10.1f}")

In [ ]:
# 9. Fine-tuning setup
#
# The fused model has good initialization from JuliaSLM but needs fine-tuning to:
# 1. Adapt the expanded dimensions (256→512) from zero/noise to useful features
# 2. Train layers 7-8 (randomly initialized)
# 3. Learn the GQA attention pattern (from MHA initialization)
#
# We use a moderate LR (lower than training from scratch) since most weights
# already have good values from JuliaSLM.

BATCH_SIZE = 64
TOKENS_PER_STEP = BATCH_SIZE * CTX  # 16,384
TOTAL_STEPS = 8000          # ~131M tokens (half the curated corpus)
WARMUP = 300
LR = 3e-4                   # moderate: not too aggressive for transferred weights
MIN_LR = 1e-5
EVAL_INTERVAL = 500
CHECKPOINT_INTERVAL = 2000

HF_REPO = "LisaMegaWatts/JuliaFluxGPT-fused"

print(f"Fine-tuning plan:")
print(f"  Steps: {TOTAL_STEPS:,} ({TOTAL_STEPS * TOKENS_PER_STEP / 1e6:.0f}M tokens)")
print(f"  LR: {LR} → {MIN_LR} (cosine, {WARMUP} warmup)")
print(f"  Batch: {BATCH_SIZE} × {CTX} = {TOKENS_PER_STEP:,} tok/step")
print(f"  Data epochs: {TOTAL_STEPS * TOKENS_PER_STEP / (len(train_inputs) * CTX):.2f}")

# W&B init
run = wandb.init(
    project="symbiogenesis",
    name="juliafluxgpt-slm-fusion",
    config={
        "model": "JuliaFluxGPT-fused",
        "method": "symbiogenesis_projection_fusion",
        "source": "JuliaSLM (d=256, 6L, 4H MHA, val_loss=3.54)",
        "target": "JuliaFluxGPT (d=512, 8L, 8Q/2KV GQA)",
        "d_model": config.d_model,
        "n_layers": config.n_layers,
        "n_heads": config.n_heads,
        "n_kv_heads": config.n_kv_heads,
        "vocab_size": config.vocab_size,
        "total_params": model.num_parameters,
        "batch_size": BATCH_SIZE,
        "total_steps": TOTAL_STEPS,
        "lr": LR,
        "min_lr": MIN_LR,
        "warmup": WARMUP,
        "fused_val_loss_before_ft": fused_loss,
        "orig_val_loss": orig_loss,
        "source_val_loss": 3.54,
    },
    tags=["fusion", "projection", "juliafluxgpt", "juliaslm", "symbiogenesis"],
    reinit=True,
)
print(f"\nW&B: {run.url}")

In [ ]:
# 10. Fine-tuning loop

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.95)
)

def lr_schedule(step):
    if step < WARMUP:
        return (step + 1) / max(WARMUP, 1)
    progress = (step - WARMUP) / max(TOTAL_STEPS - WARMUP, 1)
    return max(MIN_LR / LR, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

n_train = len(train_inputs)
model.train()
best_val_loss = float("inf")
best_step = 0
history = []
t_start = time.time()
step = 0

print(f"Fine-tuning JuliaFluxGPT-fused for {TOTAL_STEPS} steps...")
print(f"Precision: {amp_dtype}, batch={BATCH_SIZE}, lr={LR}")

while step < TOTAL_STEPS:
    perm = torch.randperm(n_train)
    for i in range(0, n_train, BATCH_SIZE):
        if step >= TOTAL_STEPS:
            break

        idx = perm[i:i+BATCH_SIZE]
        batch_in = train_inputs[idx].to(device)
        batch_tgt = train_labels[idx].to(device)

        with torch.amp.autocast("cuda", enabled=True, dtype=amp_dtype):
            logits = model(batch_in)
            B, T, V = logits.shape
            loss = F.cross_entropy(logits.reshape(B*T, V), batch_tgt.reshape(B*T))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        scheduler.step()

        if step % 50 == 0:
            elapsed = time.time() - t_start
            tok_s = (step + 1) * TOKENS_PER_STEP / max(elapsed, 1)
            wandb.log({
                "train/loss": loss.item(),
                "train/lr": scheduler.get_last_lr()[0],
                "train/tokens_per_sec": tok_s,
            }, step=step)

        if step > 0 and step % EVAL_INTERVAL == 0:
            val_loss, val_ppl = evaluate(model, val_inputs, val_labels)
            history.append((step, val_loss, val_ppl))
            marker = " ** NEW BEST **" if val_loss < best_val_loss else ""
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_step = step
                torch.save({
                    "model_state_dict": model.state_dict(),
                    "config": {
                        "d_model": config.d_model, "n_layers": config.n_layers,
                        "n_heads": config.n_heads, "n_kv_heads": config.n_kv_heads,
                        "head_dim": config.head_dim, "context_length": CTX,
                        "vocab_size": config.vocab_size, "weight_tying": True,
                    },
                    "step": step, "val_loss": val_loss, "val_ppl": val_ppl,
                    "n_params": model.num_parameters,
                    "fusion_source": "JuliaSLM (d=256, 6L, val_loss=3.54)",
                }, "juliaflux_fused_best.pt")
            wandb.log({"val/loss": val_loss, "val/perplexity": val_ppl}, step=step)
            elapsed = time.time() - t_start
            print(f"  [step {step:5d}] val_loss={val_loss:.4f} ppl={val_ppl:.1f} "
                  f"lr={scheduler.get_last_lr()[0]:.2e} ({elapsed:.0f}s){marker}")
            model.train()

        if step > 0 and step % CHECKPOINT_INTERVAL == 0:
            torch.save(model.state_dict(), f"juliaflux_fused_step{step}.pt")

        step += 1

# Final eval
final_loss, final_ppl = evaluate(model, val_inputs, val_labels)
history.append((step, final_loss, final_ppl))
if final_loss < best_val_loss:
    best_val_loss = final_loss
    best_step = step
    torch.save({
        "model_state_dict": model.state_dict(),
        "config": {
            "d_model": config.d_model, "n_layers": config.n_layers,
            "n_heads": config.n_heads, "n_kv_heads": config.n_kv_heads,
            "head_dim": config.head_dim, "context_length": CTX,
            "vocab_size": config.vocab_size, "weight_tying": True,
        },
        "step": step, "val_loss": final_loss, "val_ppl": final_ppl,
        "n_params": model.num_parameters,
        "fusion_source": "JuliaSLM (d=256, 6L, val_loss=3.54)",
    }, "juliaflux_fused_best.pt")

elapsed = time.time() - t_start
wandb.log({"val/final_loss": best_val_loss, "val/final_ppl": math.exp(min(best_val_loss, 20.0))})
print(f"\nFine-tuning complete ({elapsed:.0f}s)")
print(f"Best: val_loss={best_val_loss:.4f} ppl={math.exp(min(best_val_loss, 20.0)):.1f} at step {best_step}")

In [ ]:
# 11. Results comparison

print("\n" + "=" * 65)
print("FUSION RESULTS — Symbiogenesis Projection Transfer")
print("=" * 65)
print(f"\n{'Model':<35} {'Params':>10} {'Val Loss':>10} {'PPL':>8}")
print("-" * 65)
print(f"{'JuliaSLM (source)':<35} {'5,040,000':>10} {'3.5400':>10} {'34.5':>8}")
print(f"{'JuliaFluxGPT (dirty, original)':<35} {'22,790,000':>10} {orig_loss:>10.4f} {orig_ppl:>8.1f}")
print(f"{'Fused (before fine-tune)':<35} {'22,790,000':>10} {fused_loss:>10.4f} {fused_ppl:>8.1f}")
print(f"{'Fused (after fine-tune) **':<35} {'22,790,000':>10} {best_val_loss:>10.4f} {math.exp(min(best_val_loss,20)):>8.1f}")
print(f"\n** Best at step {best_step}")

print(f"\nScaling law context (curated data, BPE vocab=2000, ctx=256):")
print(f"  SymbioSLM      4.07M  →  3.62")
print(f"  MonarchSLM     4.98M  →  3.65")
print(f"  JuliaSLM       5.04M  →  3.54")
print(f"  SymbioGPT-10M 11.05M  →  3.56")
print(f"  JuliaFluxGPT* 22.79M  →  {best_val_loss:.2f}  (* fused from JuliaSLM)")

print(f"\nTraining history:")
for s, loss, ppl in history:
    print(f"  step {s:5d}: loss={loss:.4f} ppl={ppl:.1f}")

wandb.finish()

In [ ]:
# 12. Generate sample text

# Download BPE tokenizer
hf_hub_download(repo_id=DATA_REPO, filename="tokenizer/vocab.json", local_dir=".")
hf_hub_download(repo_id=DATA_REPO, filename="tokenizer/merges.txt", local_dir=".")

sys.path.insert(0, "/content/SymbioGPT")
try:
    from tokenizer import BPETokenizer
    tok = BPETokenizer("tokenizer/vocab.json", "tokenizer/merges.txt")
except ImportError:
    import json
    class SimpleBPETokenizer:
        def __init__(self, vocab_path):
            with open(vocab_path) as f:
                self.encoder = json.load(f)
            self.decoder = {v: k for k, v in self.encoder.items()}
        def decode(self, ids):
            return ''.join(self.decoder.get(i, '?') for i in ids)
    tok = SimpleBPETokenizer("tokenizer/vocab.json")

model.eval()
prompts = ["The nature of", "Philosophy teaches", "In the beginning"]

for prompt_text in prompts:
    # Encode prompt
    tokens = [tok.encoder.get(c, 0) for c in prompt_text]  # simple char-level fallback
    if not tokens:
        tokens = [1]
    input_ids = torch.tensor([tokens[-CTX:]], device=device)
    
    # Generate
    generated = list(tokens)
    with torch.no_grad():
        for _ in range(100):
            logits = model(input_ids)
            next_logits = logits[0, -1] / 0.8  # temperature
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            generated.append(next_token)
            input_ids = torch.tensor([generated[-CTX:]], device=device)
    
    text = tok.decode(generated)
    print(f"\n--- Prompt: '{prompt_text}' ---")
    print(text[:300])

In [ ]:
# 13. Upload to HuggingFace

hf_api = HfApi()
try:
    create_repo(HF_REPO, exist_ok=True)
    
    # Upload best checkpoint
    hf_api.upload_file(
        path_or_fileobj="juliaflux_fused_best.pt",
        path_in_repo="juliaflux_fused_best.pt",
        repo_id=HF_REPO,
        commit_message=f"Fused best: val_loss={best_val_loss:.4f} ppl={math.exp(min(best_val_loss,20)):.1f} at step {best_step}"
    )
    
    # Upload model definition
    hf_api.upload_file(
        path_or_fileobj="juliaflux_model.py",
        path_in_repo="juliaflux_model.py",
        repo_id=HF_REPO,
        commit_message="Model definition"
    )
    
    print(f"Uploaded to: https://huggingface.co/{HF_REPO}")
except Exception as e:
    print(f"HF upload failed: {e}")

print("\nDone!")